# BM25 using CSC Matrix
- Compressed Sparse Column (CSC) formatted Matrix
  - column-based compression for the almost zero-filled matrix

In [88]:
import os
import numpy as np

import marisa_trie
import pydantic
from scipy.sparse import csc_matrix, csr_matrix

## CSC Matrix Examples

In [89]:
from scipy.sparse import csc_matrix
import numpy as np

# (1) csc matrix from dense array
dense = np.array([[0, 0, 1],
                  [4, 0, 0],
                  [0, 5, 6]])
A = csc_matrix(dense)

# (2) csc matrix from (data, (row, col)) format
data = [1, 4, 5, 6]  # non-zero entries
rows = [0, 1, 2, 2]  # row indices of non-zero entries
cols = [2, 0, 1, 2]  # column indices of non-zero entries
B = csc_matrix((data, (rows, cols)), shape=(3, 3))

print(A.toarray())
print(B.toarray())


[[0 0 1]
 [4 0 0]
 [0 5 6]]
[[0 0 1]
 [4 0 0]
 [0 5 6]]


In [90]:
(
    A.data,  # non-zero entries
    A.indices,  # row indices of non-zero entries
    A.indptr  # column pointer (index in data/indices where each column starts),
                # indptr has length n_cols + 1
                # e.g., for 3 columns, indptr has length 4
                # indptr[0] = 0 (start of col 0)
                # indptr[1] = 1 (start of col 1)
                # indptr[2] = 2 (start of col 2)
                # indptr[3] = 4 (end of col 2, total number of non-zero entries)
                # so col 0 has 1 entry, col 1 has 1 entry, col 2 has 2 entries
                # thus indptr = [0, 1, 2, 4]
                # see https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csc_matrix.html
                # for more details
                # this is useful for efficient column slicing and matrix-vector products
                # e.g., A[:, 2] can be accessed directly using indptr[2] to indptr[3]
                # without scanning the entire data array
                # this is different from CSR format which is row-based
                # see https://en.wikipedia.org/wiki/Sparse_matrix#Compressed_sparse_column_(CSC)
                # for more info
                # in summary, indptr helps locate the start and end of each column in the data/indices arrays
                # enabling efficient column operations
                # this is particularly useful in applications like BM25 where column operations are common
                # e.g., computing term frequencies across documents
                # thus understanding indptr is crucial for working with CSC matrices effectively
                # especially in information retrieval contexts
                # indptr[j] to indptr[j+1] gives the range of non-zero entries for column j, 0 <= j < n_cols
                # data[indptr[j]:indptr[j+1]] gives the actual non-zero values in column j
                # indices[indptr[j]:indptr[j+1]] gives the corresponding row indices for those values
)

(array([4, 5, 1, 6]),
 array([1, 2, 0, 2], dtype=int32),
 array([0, 1, 2, 4], dtype=int32))

In [91]:
bool((A.data == B.data).all() and (A.indices == B.indices).all() and (A.indptr == B.indptr).all())

True

In [92]:
# Example documents
documents = [
    {"title": "Cat Facts", "text": "Cats are curious animals."},
    {"title": "Dog Facts", "text": "Dogs are loyal and friendly."},
    {"title": "Bird Facts", "text": "Birds can fly and sing."}
]

n_docs = len(documents)

In [93]:
# Build vocabulary and trie
all_tokens = set()
for doc in documents:
    all_tokens.update(doc["text"].lower().split())


In [94]:

trie = marisa_trie.Trie(sorted(all_tokens))
n_vocab = len(trie)

token2id = {token: idx for idx, token in enumerate(trie)}

shape = (n_docs, n_vocab)

In [95]:
shape, trie.items(), token2id

((3, 12),
 [('and', 8),
  ('animals.', 9),
  ('are', 4),
  ('can', 10),
  ('cats', 11),
  ('curious', 5),
  ('fly', 6),
  ('friendly.', 7),
  ('birds', 0),
  ('dogs', 1),
  ('loyal', 2),
  ('sing.', 3)],
 {'and': 0,
  'animals.': 1,
  'are': 2,
  'can': 3,
  'cats': 4,
  'curious': 5,
  'fly': 6,
  'friendly.': 7,
  'birds': 8,
  'dogs': 9,
  'loyal': 10,
  'sing.': 11})

In [96]:
def stream_docs(columnar_texts, token2id ):
    for text in columnar_texts:
        tokens = text.lower().split()  # TODO: improve tokenization
        token_ids = np.array([token2id[token] for token in tokens if token in token2id], dtype=np.int32)
        yield token_ids

In [97]:
# df & nnz_total
df = np.zeros(n_vocab, dtype=np.int32)
doc_len = np.zeros(n_docs, dtype=np.int32)
nnz_total = 0  # total count of unique term occurrences in a document

field_name = "text"

columnar_posting = [doc[field_name] for doc in documents]

for d, terms in enumerate(stream_docs(columnar_posting, trie)):  # terms: np.array of vocab ids (duplicates allowed)
    doc_len[d] = len(terms)
    uniq = np.unique(terms)
    df[uniq] += 1
    nnz_total += uniq.size


In [98]:
nnz_total

14

In [99]:
# idf
idf = np.log((n_docs - df + 0.5) / (df + 0.5))
idf = np.maximum(idf, 0)
idf

array([0.51082562, 0.51082562, 0.51082562, 0.51082562, 0.        ,
       0.51082562, 0.51082562, 0.51082562, 0.        , 0.51082562,
       0.51082562, 0.51082562])

In [100]:
# indptr
indptr = np.empty(n_vocab + 1, dtype=np.int32)
indptr[0] = 0
np.cumsum(df, out=indptr[1:])
indptr

array([ 0,  1,  2,  3,  4,  6,  7,  8,  9, 11, 12, 13, 14], dtype=int32)

In [101]:
indices = np.memmap("indices.bin", dtype=np.int32,   mode="w+", shape=(nnz_total,))
data    = np.memmap("data.bin",    dtype=np.float32, mode="w+", shape=(nnz_total,))
offset  = indptr.copy()

# BM25 준비
k1, b = 1.2, 0.75
avgdl = float(doc_len.mean())
# idf   = ...  # 길이 n_vocab, float32: idf[t] = log((N - df[t] + 0.5)/(df[t] + 0.5) + 1), N=n_docs

columnar_posting = [doc[field_name] for doc in documents]

print(offset.shape)
print(f"offset samples: {offset[:10]}")

for d, terms in enumerate(stream_docs(columnar_posting, trie)):
# for d, terms in enumerate(stream_docs(columnar_posting, token2id)):
    uniq, counts = np.unique(terms, return_counts=True)
    dl = float(doc_len[d])
    tf = counts.astype(np.float32)
    bm25 = idf[uniq] * (tf*(k1+1.0)) / (tf + k1*(1.0 - b + b*dl/avgdl))

    # start = offset[uniq]
    # end   = start + counts.size
    # indices[start:end] = [d] * counts.size
    # data[start:end]    = bm25
    # offset[uniq]      += counts.size

    # 각 단어 컬럼에 바로 써넣음 → indices=문서ID, data=점수
    for v, s in zip(uniq.astype(np.int32), bm25):
        pos = offset[v]
        indices[pos] = d
        data[pos]    = s
        offset[v]   += 1

from scipy.sparse import csc_matrix
X = csc_matrix((data, indices, indptr), shape=(n_docs, n_vocab))


(13,)
offset samples: [ 0  1  2  3  4  6  7  8  9 11]


In [102]:
X.toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.5425321 , 0.        , 0.        , 0.        , 0.5425321 ,
        0.        , 0.5425321 ],
       [0.        , 0.49632272, 0.49632272, 0.        , 0.        ,
        0.        , 0.        , 0.49632272, 0.        , 0.        ,
        0.        , 0.        ],
       [0.49632272, 0.        , 0.        , 0.49632272, 0.        ,
        0.        , 0.49632272, 0.        , 0.        , 0.        ,
        0.49632272, 0.        ]], dtype=float32)

In [103]:
data, indices, indptr

(memmap([0.49632272, 0.49632272, 0.49632272, 0.49632272, 0.        ,
         0.        , 0.5425321 , 0.49632272, 0.49632272, 0.        ,
         0.        , 0.5425321 , 0.49632272, 0.5425321 ], dtype=float32),
 memmap([2, 1, 1, 2, 0, 1, 0, 2, 1, 1, 2, 0, 2, 0], dtype=int32),
 array([ 0,  1,  2,  3,  4,  6,  7,  8,  9, 11, 12, 13, 14], dtype=int32))

In [104]:
len(indices), len(data), len(indptr)

(14, 14, 13)

In [105]:
X.shape

(3, 12)

In [106]:
indptr.shape, indptr

((13,),
 array([ 0,  1,  2,  3,  4,  6,  7,  8,  9, 11, 12, 13, 14], dtype=int32))

In [107]:
data.shape, indices.shape, nnz_total

((14,), (14,), 14)

In [108]:
token2id

{'and': 0,
 'animals.': 1,
 'are': 2,
 'can': 3,
 'cats': 4,
 'curious': 5,
 'fly': 6,
 'friendly.': 7,
 'birds': 8,
 'dogs': 9,
 'loyal': 10,
 'sing.': 11}

In [109]:
trie.items()

[('and', 8),
 ('animals.', 9),
 ('are', 4),
 ('can', 10),
 ('cats', 11),
 ('curious', 5),
 ('fly', 6),
 ('friendly.', 7),
 ('birds', 0),
 ('dogs', 1),
 ('loyal', 2),
 ('sing.', 3)]

## scoring term-query scores of document '0'

In [110]:
X[0, trie["curious"]], X[0, trie["cats"]]

(np.float32(0.5425321), np.float32(0.5425321))

## Searching Top-k documents for individual query term

In [111]:
def search_top_k_docs_for_term(X, term_id, k=3):
    """
    Search top-k documents for a given term using CSC matrix.
    
    Args:
        X: CSC matrix (n_docs, n_vocab) with BM25 scores
        term_id: vocabulary term ID
        k: number of top documents to return
    
    Returns:
        tuple: (doc_ids, scores) sorted by score descending
    """
    # Extract the column for this term (all documents' scores for this term)
    term_column = X[:, term_id]
    
    # Convert to dense array for easier manipulation
    scores = term_column.toarray().flatten()
    
    # Get non-zero indices and their scores
    non_zero_mask = scores > 0
    doc_ids = np.where(non_zero_mask)[0]
    doc_scores = scores[non_zero_mask]
    
    # Sort by scores in descending order
    sorted_indices = np.argsort(doc_scores)[::-1]
    
    # Return top-k
    top_k = min(k, len(sorted_indices))
    top_doc_ids = doc_ids[sorted_indices[:top_k]]
    top_scores = doc_scores[sorted_indices[:top_k]]
    
    return top_doc_ids, top_scores

# Example: Search for top documents containing "curious"
term_id = trie["curious"]
top_docs, top_scores = search_top_k_docs_for_term(X, term_id, k=3)

print(f"Top documents for term 'curious' (ID: {term_id}):")
for doc_id, score in zip(top_docs, top_scores):
    print(f"  Document {doc_id}: score = {score:.4f}")
    print(f"    Content: {documents[doc_id]['text']}")

Top documents for term 'curious' (ID: 5):
  Document 0: score = 0.5425
    Content: Cats are curious animals.


In [112]:
def search_top_k_docs_for_term_efficient(X, term_id, k=3):
    """
    More efficient version that directly uses CSC matrix internals.
    Avoids converting to dense array.
    
    Args:
        X: CSC matrix (n_docs, n_vocab) with BM25 scores
        term_id: vocabulary term ID
        k: number of top documents to return
    
    Returns:
        tuple: (doc_ids, scores) sorted by score descending
    """
    # Use CSC matrix structure directly
    start_idx = X.indptr[term_id]
    end_idx = X.indptr[term_id + 1]
    
    # Extract document IDs and scores for this term
    doc_ids = X.indices[start_idx:end_idx]
    scores = X.data[start_idx:end_idx]
    
    # Sort by scores in descending order
    sorted_indices = np.argsort(scores)[::-1]
    
    # Return top-k
    top_k = min(k, len(sorted_indices))
    top_doc_ids = doc_ids[sorted_indices[:top_k]]
    top_scores = scores[sorted_indices[:top_k]]
    
    return top_doc_ids, top_scores

# Test both methods with different terms
test_terms = ["curious", "cats", "dogs", "birds"]

for term in test_terms:
    if term in token2id:
        term_id = trie[term]
        print(f"\n=== Term: '{term}' (ID: {term_id}) ===")
        
        # Method 1: Using dense conversion
        top_docs1, top_scores1 = search_top_k_docs_for_term(X, term_id, k=2)
        
        # Method 2: Using CSC internals directly
        top_docs2, top_scores2 = search_top_k_docs_for_term_efficient(X, term_id, k=2)
        
        print(f"Dense method results: docs={top_docs1}, scores={top_scores1}")
        print(f"CSC method results:   docs={top_docs2}, scores={top_scores2}")
        
        # Show document contents
        for doc_id, score in zip(top_docs2, top_scores2):
            print(f"  Doc {doc_id} (score: {score:.4f}): {documents[doc_id]['text']}")
    else:
        print(f"Term '{term}' not found in vocabulary")


=== Term: 'curious' (ID: 5) ===
Dense method results: docs=[0], scores=[0.5425321]
CSC method results:   docs=[0], scores=[0.5425321]
  Doc 0 (score: 0.5425): Cats are curious animals.

=== Term: 'cats' (ID: 11) ===
Dense method results: docs=[0], scores=[0.5425321]
CSC method results:   docs=[0], scores=[0.5425321]
  Doc 0 (score: 0.5425): Cats are curious animals.

=== Term: 'dogs' (ID: 1) ===
Dense method results: docs=[1], scores=[0.49632272]
CSC method results:   docs=[1], scores=[0.49632272]
  Doc 1 (score: 0.4963): Dogs are loyal and friendly.

=== Term: 'birds' (ID: 0) ===
Dense method results: docs=[2], scores=[0.49632272]
CSC method results:   docs=[2], scores=[0.49632272]
  Doc 2 (score: 0.4963): Birds can fly and sing.


In [113]:
def search_top_k_docs_multi_terms(X, term_ids, k=3, aggregation='sum'):
    """
    Search top-k documents for multiple terms (query with multiple words).
    
    Args:
        X: CSC matrix (n_docs, n_vocab) with BM25 scores
        term_ids: list of term IDs
        k: number of top documents to return
        aggregation: 'sum' or 'max' - how to combine scores from multiple terms
    
    Returns:
        tuple: (doc_ids, scores) sorted by score descending
    """
    n_docs = X.shape[0]
    combined_scores = np.zeros(n_docs, dtype=np.float32)
    
    for term_id in term_ids:
        # Extract scores for this term
        start_idx = X.indptr[term_id]
        end_idx = X.indptr[term_id + 1]
        
        doc_ids = X.indices[start_idx:end_idx]
        scores = X.data[start_idx:end_idx]
        
        # Combine scores
        if aggregation == 'sum':
            combined_scores[doc_ids] += scores
        elif aggregation == 'max':
            combined_scores[doc_ids] = np.maximum(combined_scores[doc_ids], scores)
    
    # Get top-k documents
    non_zero_mask = combined_scores > 0
    doc_ids = np.where(non_zero_mask)[0]
    doc_scores = combined_scores[non_zero_mask]
    
    # Sort by scores in descending order
    sorted_indices = np.argsort(doc_scores)[::-1]
    
    # Return top-k
    top_k = min(k, len(sorted_indices))
    top_doc_ids = doc_ids[sorted_indices[:top_k]]
    top_scores = doc_scores[sorted_indices[:top_k]]
    
    return top_doc_ids, top_scores

# Example: Search for documents containing both "cats" and "curious" 
query_terms = ["cats", "curious"]
term_ids = [trie[term] for term in query_terms if term in token2id]

print(f"Multi-term search for: {query_terms}")
print(f"Term IDs: {term_ids}")

top_docs, top_scores = search_top_k_docs_multi_terms(X, term_ids, k=3, aggregation='sum')

print(f"\nTop documents (sum aggregation):")
for doc_id, score in zip(top_docs, top_scores):
    print(f"  Document {doc_id}: score = {score:.4f}")
    print(f"    Content: {documents[doc_id]['text']}")

# Also try max aggregation
top_docs_max, top_scores_max = search_top_k_docs_multi_terms(X, term_ids, k=3, aggregation='max')
print(f"\nTop documents (max aggregation):")
for doc_id, score in zip(top_docs_max, top_scores_max):
    print(f"  Document {doc_id}: score = {score:.4f}")
    print(f"    Content: {documents[doc_id]['text']}")

Multi-term search for: ['cats', 'curious']
Term IDs: [11, 5]

Top documents (sum aggregation):
  Document 0: score = 1.0851
    Content: Cats are curious animals.

Top documents (max aggregation):
  Document 0: score = 0.5425
    Content: Cats are curious animals.


## 개선된 방법들:
1. Dictionary 기반 방법 (search_top_k_docs_multi_terms_efficient)
- n_docs 크기 배열 대신 딕셔너리 사용
- 실제로 점수가 있는 문서들만 추적
- 메모리 사용량: O(관련 문서 수) vs O(전체 문서 수)
2. Sparse Operations 방법 (search_top_k_docs_multi_terms_sparse)
- scipy sparse matrix의 내장 연산 활용
- X[:, term_ids].sum(axis=1) 또는 maximum() 사용
- 가장 효율적이고 간결한 코드

## 장점 비교:
### Dictionary 방법:
- 메모리 효율적 (sparse한 경우)
- 구현이 직관적
- 큰 문서 컬렉션에서 유리

### Sparse Operations 방법:
- 가장 빠름 (scipy 최적화된 연산)
- 코드가 가장 간결
- NumPy/SciPy의 벡터화 이점

실제 대용량 데이터에서는 **Method 3 (sparse operations)**가 가장 효율적

In [114]:
def search_top_k_docs_multi_terms_efficient(X, term_ids, k=3, aggregation='sum'):
    """
    Memory-efficient version using dictionary to track only relevant documents.
    
    Args:
        X: CSC matrix (n_docs, n_vocab) with BM25 scores
        term_ids: list of term IDs
        k: number of top documents to return
        aggregation: 'sum' or 'max' - how to combine scores from multiple terms
    
    Returns:
        tuple: (doc_ids, scores) sorted by score descending
    """
    # Use dictionary to track only documents that have at least one term
    doc_scores = {}
    
    for term_id in term_ids:
        # Extract scores for this term
        start_idx = X.indptr[term_id]
        end_idx = X.indptr[term_id + 1]
        
        doc_ids = X.indices[start_idx:end_idx]
        scores = X.data[start_idx:end_idx]
        
        # Combine scores only for relevant documents
        for doc_id, score in zip(doc_ids, scores):
            if aggregation == 'sum':
                doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + score
            elif aggregation == 'max':
                doc_scores[doc_id] = max(doc_scores.get(doc_id, 0.0), score)
    
    if not doc_scores:
        return np.array([]), np.array([])
    
    # Sort by scores in descending order
    sorted_items = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Return top-k
    top_k = min(k, len(sorted_items))
    top_doc_ids = np.array([item[0] for item in sorted_items[:top_k]], dtype=np.int32)
    top_scores = np.array([item[1] for item in sorted_items[:top_k]], dtype=np.float32)
    
    return top_doc_ids, top_scores


def search_top_k_docs_multi_terms_sparse(X, term_ids, k=3, aggregation='sum'):
    """
    Even more efficient version using sparse operations directly.
    
    Args:
        X: CSC matrix (n_docs, n_vocab) with BM25 scores
        term_ids: list of term IDs
        k: number of top documents to return
        aggregation: 'sum' or 'max' - how to combine scores from multiple terms
    
    Returns:
        tuple: (doc_ids, scores) sorted by score descending
    """
    if not term_ids:
        return np.array([]), np.array([])
    
    # Extract columns for all terms and sum/max them
    if aggregation == 'sum':
        # Sum the columns for all terms
        # the type of combined is numpy matrix
        combined = X[:, term_ids].sum(axis=1)
        print(f"#### combined type (sum): {type(combined)}")
    elif aggregation == 'max':
        # Take element-wise maximum across columns
        # the type of combined-init is csc_matrix
        combined = X[:, term_ids[0]]
        print(f"#### combined type (max) init: {type(combined)}")
        # the type of combined is csr_matrix
        for term_id in term_ids[1:]:
            combined = combined.maximum(X[:, term_id])
        print(f"#### combined type (max): {type(combined)}")
    else:
        raise ValueError("aggregation must be 'sum' or 'max'")
    
    # Convert to 1D array and get non-zero indices
    # combined_flat = np.asarray(combined).flatten()
    if hasattr(combined, 'toarray') and isinstance(combined, csr_matrix):
        # when combined is sparse matrix
        combined_flat = combined.toarray().flatten()
    else:
        # when combined is np array
        combined_flat = np.asarray(combined).flatten()
    
    non_zero_indices = np.where(combined_flat > 0)[0]
    
    if len(non_zero_indices) == 0:
        return np.array([]), np.array([])
    
    # Get scores for non-zero documents
    scores = combined_flat[non_zero_indices]
    
    # Sort by scores in descending order
    sorted_indices = np.argsort(scores)[::-1]
    
    # Return top-k
    top_k = min(k, len(sorted_indices))
    top_doc_ids = non_zero_indices[sorted_indices[:top_k]]
    top_scores = scores[sorted_indices[:top_k]]
    
    return top_doc_ids, top_scores

In [115]:
# Compare all three methods
query_terms = ["cats", "curious"]
term_ids = [trie[term] for term in query_terms if term in token2id]

print(f"Comparing methods for query: {query_terms}")
print(f"Term IDs: {term_ids}")
print(f"Matrix shape: {X.shape}")

# Method 1: Original (using full n_docs array)
top_docs1, top_scores1 = search_top_k_docs_multi_terms(X, term_ids, k=3, aggregation='sum')

# Method 2: Dictionary-based (memory efficient)
top_docs2, top_scores2 = search_top_k_docs_multi_terms_efficient(X, term_ids, k=3, aggregation='sum')

# Method 3: Sparse operations (most efficient)
top_docs3, top_scores3 = search_top_k_docs_multi_terms_sparse(X, term_ids, k=3, aggregation='sum')

print(f"\n=== Results Comparison ===")
print(f"Method 1 (sum, full array):  docs={top_docs1}, scores={top_scores1}")
print(f"Method 2 (sum, dictionary):  docs={top_docs2}, scores={top_scores2}")
print(f"Method 3 (sum, sparse ops):  docs={top_docs3}, scores={top_scores3}")

print(f"\n=== Document Contents ===")
for doc_id, score in zip(top_docs3, top_scores3):
    print(f"  Doc {doc_id} (score: {score:.4f}): {documents[doc_id]['text']}")

# Test with different aggregation
print(f"\n=== Max Aggregation Test ===")
top_docs_max1, top_scores_max1 = search_top_k_docs_multi_terms(X, term_ids, k=3, aggregation='max')
top_docs_max2, top_scores_max2 = search_top_k_docs_multi_terms_efficient(X, term_ids, k=3, aggregation='max')
top_docs_max3, top_scores_max3 = search_top_k_docs_multi_terms_sparse(X, term_ids, k=3, aggregation='max')

print(f"Method 1 (max, full array):  docs={top_docs_max1}, scores={top_scores_max1}")
print(f"Method 2 (max, dictionary):  docs={top_docs_max2}, scores={top_scores_max2}")
print(f"Method 3 (max, sparse ops):  docs={top_docs_max3}, scores={top_scores_max3}")

Comparing methods for query: ['cats', 'curious']
Term IDs: [11, 5]
Matrix shape: (3, 12)
#### combined type (sum): <class 'numpy.matrix'>

=== Results Comparison ===
Method 1 (sum, full array):  docs=[0], scores=[1.0850642]
Method 2 (sum, dictionary):  docs=[0], scores=[1.0850642]
Method 3 (sum, sparse ops):  docs=[0], scores=[1.0850642]

=== Document Contents ===
  Doc 0 (score: 1.0851): Cats are curious animals.

=== Max Aggregation Test ===
#### combined type (max) init: <class 'scipy.sparse._csc.csc_matrix'>
#### combined type (max): <class 'scipy.sparse._csr.csr_matrix'>
Method 1 (max, full array):  docs=[0], scores=[0.5425321]
Method 2 (max, dictionary):  docs=[0], scores=[0.5425321]
Method 3 (max, sparse ops):  docs=[0], scores=[0.5425321]


In [116]:
import time

def benchmark_methods(X, term_ids, k=3, num_runs=1000):
    """
    Benchmark the three different methods for multi-term search.
    """
    print(f"Benchmarking {num_runs} runs...")
    
    # Method 1: Full array
    start_time = time.time()
    for _ in range(num_runs):
        search_top_k_docs_multi_terms(X, term_ids, k=k)
    time1 = time.time() - start_time
    
    # Method 2: Dictionary
    start_time = time.time()
    for _ in range(num_runs):
        search_top_k_docs_multi_terms_efficient(X, term_ids, k=k)
    time2 = time.time() - start_time
    
    # Method 3: Sparse operations
    start_time = time.time()
    for _ in range(num_runs):
        search_top_k_docs_multi_terms_sparse(X, term_ids, k=k)
    time3 = time.time() - start_time
    
    print(f"\nBenchmark Results ({num_runs} runs):")
    print(f"Method 1 (full array):  {time1:.4f}s ({time1/num_runs*1000:.2f}ms per run)")
    print(f"Method 2 (dictionary):  {time2:.4f}s ({time2/num_runs*1000:.2f}ms per run)")
    print(f"Method 3 (sparse ops):  {time3:.4f}s ({time3/num_runs*1000:.2f}ms per run)")
    
    print(f"\nSpeedup vs Method 1:")
    print(f"Method 2: {time1/time2:.2f}x faster")
    print(f"Method 3: {time1/time3:.2f}x faster")
    
    return time1, time2, time3

# Run benchmark
query_terms = ["cats", "curious"]
term_ids = [trie[term] for term in query_terms if term in token2id]

# For small dataset, use fewer runs
benchmark_methods(X, term_ids, k=3, num_runs=100)

Benchmarking 100 runs...
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matrix'>
#### combined type (sum): <class 'numpy.matr

(0.004442691802978516, 0.0012440681457519531, 0.023131847381591797)